In [4]:
# ============================================================
# SwiftRide Case Study
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# ============================================================
# 1. Generate Synthetic Data
# ============================================================

def generate_swiftride_data(n_trips=10000000):
    """Generate realistic ride-hailing trip data"""
    
    # Date range: last 6 months
    end_date = datetime(2025, 3, 31)
    start_date = end_date - timedelta(days=180)
    
    # Helper functions
    def random_date(start, end):
        return start + timedelta(seconds=np.random.randint(0, int((end - start).total_seconds())))
    
    # Predefined lists
    traffic_conditions = ['Light', 'Moderate', 'Heavy']
    traffic_probs = [0.3, 0.5, 0.2]
    weather_conditions = ['Clear', 'Rain', 'Fog', 'Storm']
    weather_probs = [0.6, 0.25, 0.1, 0.05]
    payment_methods = ['Card', 'Wallet', 'Cash']
    payment_probs = [0.5, 0.3, 0.2]
    
    # Generate data
    data = []
    for i in range(n_trips):
        # Trip timing
        trip_start = random_date(start_date, end_date)
        hour = trip_start.hour
        day_of_week = trip_start.weekday()
        
        # Weather and traffic influenced by hour/day
        if hour in [7,8,9,17,18,19]:
            traffic = np.random.choice(traffic_conditions, p=[0.1, 0.5, 0.4])
        else:
            traffic = np.random.choice(traffic_conditions, p=[0.5, 0.4, 0.1])
        
        weather = np.random.choice(weather_conditions, p=weather_probs)
        
        # Surge multiplier based on hour, day, weather, traffic
        base_surge = 1.0
        if hour in [22,23,0,1,2]:
            base_surge += 0.5
        if day_of_week >= 5:  # weekend
            base_surge += 0.3
        if weather in ['Rain', 'Storm']:
            base_surge += 0.4
        if traffic == 'Heavy':
            base_surge += 0.3
        surge = round(np.random.uniform(base_surge, base_surge + 0.5), 1)
        
        # Trip distance (km) - lognormal
        distance = round(np.random.lognormal(mean=1.5, sigma=0.8), 1)
        # Duration (seconds) = distance * (speed factor) + traffic/weather delay
        speed_kmh = np.random.normal(25, 5)
        duration = int((distance / speed_kmh) * 3600)
        if traffic == 'Heavy':
            duration *= 1.4
        if weather in ['Rain', 'Storm']:
            duration *= 1.2
        duration = min(max(duration, 180), 7200)  # between 3 min and 2 hours
        
        # Fare: base $2 + $1.5 per km + surge multiplier
        fare = round(2 + 1.5 * distance + (distance * 0.5 * (surge - 1)), 2)
        
        # Driver and rider ratings (influenced by conditions)
        driver_rating = 5 - abs(np.random.normal(0.5, 0.3))
        if traffic == 'Heavy':
            driver_rating -= 0.3
        if weather in ['Rain', 'Storm']:
            driver_rating -= 0.2
        driver_rating = round(min(max(driver_rating, 1), 5), 1)
        
        rider_rating = 5 - abs(np.random.normal(0.4, 0.2))
        rider_rating = round(min(max(rider_rating, 1), 5), 1)
        
        # Driver tenure (days) - exponential
        driver_tenure = int(np.random.exponential(90)) + 1
        # Rider tenure
        rider_tenure = int(np.random.exponential(120)) + 1
        
        # Payment method
        payment = np.random.choice(payment_methods, p=payment_probs)
        
        data.append([
            f"T{i+1:06d}", f"R{np.random.randint(1,50000):05d}", f"D{np.random.randint(1,20000):05d}",
            trip_start, trip_start + timedelta(seconds=duration),
            distance, duration, fare, surge,
            driver_rating, rider_rating, payment,
            traffic, weather, day_of_week, hour, driver_tenure, rider_tenure
        ])
    
    columns = ['trip_id', 'rider_id', 'driver_id', 'trip_start_time', 'trip_end_time',
               'trip_distance', 'trip_duration', 'fare_amount', 'surge_multiplier',
               'driver_rating', 'rider_rating', 'payment_method',
               'traffic_condition', 'weather', 'day_of_week', 'hour_of_day',
               'driver_tenure_days', 'rider_tenure_days']
    
    df = pd.DataFrame(data, columns=columns)
    return df

# Generate data
print("Generating SwiftRide dataset...")
df = generate_swiftride_data(10000000)
print(f"Dataset shape: {df.shape}")
print(df.head())

Generating SwiftRide dataset...
Dataset shape: (10000000, 18)
   trip_id rider_id driver_id     trip_start_time           trip_end_time  \
0  T000001   R00770    D02434 2025-03-22 07:40:22 2025-03-22 07:44:21.400   
1  T000002   R11395    D03557 2025-01-16 06:05:51 2025-01-16 06:08:51.000   
2  T000003   R48556    D17160 2025-01-13 04:53:06 2025-01-13 05:03:16.000   
3  T000004   R37066    D00996 2025-03-07 00:24:14 2025-03-07 00:29:56.000   
4  T000005   R39100    D08572 2024-10-14 02:46:37 2024-10-14 03:03:19.000   

   trip_distance  trip_duration  fare_amount  surge_multiplier  driver_rating  \
0            1.7          239.4         5.40               2.0            4.2   
1            0.6          180.0         3.20               2.0            4.1   
2            4.8          610.0         9.20               1.0            4.5   
3            3.1          342.0         7.58               1.6            4.4   
4            7.0         1002.0        15.30               1.8        

In [5]:
df.head()

,trip_id,rider_id,driver_id,trip_start_time,trip_end_time,trip_distance,trip_duration,fare_amount,surge_multiplier,driver_rating,rider_rating,payment_method,traffic_condition,weather,day_of_week,hour_of_day,driver_tenure_days,rider_tenure_days
0,T000001,R00770,D02434,2025-03-22 07:40:22,2025-03-22 07:44:21.400,1.7,239.4,5.40,2.0,4.2,4.8,Wallet,Heavy,Clear,5,7,37,19
1,T000002,R11395,D03557,2025-01-16 06:05:51,2025-01-16 06:08:51.000,0.6,180.0,3.20,2.0,4.1,4.6,Cash,Heavy,Rain,3,6,87,58
2,T000003,R48556,D17160,2025-01-13 04:53:06,2025-01-13 05:03:16.000,4.8,610.0,9.20,1.0,4.5,4.6,Wallet,Moderate,Clear,0,4,24,34
3,T000004,R37066,D00996,2025-03-07 00:24:14,2025-03-07 00:29:56.000,3.1,342.0,7.58,1.6,4.4,4.9,Card,Light,Clear,4,0,3,222
4,T000005,R39100,D08572,2024-10-14 02:46:37,2024-10-14 03:03:19.000,7.0,1002.0,15.30,1.8,4.4,4.5,Cash,Moderate,Clear,0,2,70,107


In [6]:
df.to_parquet("swiftride_10M.parquet")

In [11]:
import fastparquet

In [13]:
df = pd.read_parquet("swiftride_10M.parquet", engine='fastparquet')
df

,trip_id,rider_id,driver_id,trip_start_time,trip_end_time,trip_distance,trip_duration,fare_amount,surge_multiplier,driver_rating,rider_rating,payment_method,traffic_condition,weather,day_of_week,hour_of_day,driver_tenure_days,rider_tenure_days
0,T000001,R00770,D02434,2025-03-22 07:40:22,2025-03-22 07:44:21.400,1.7,239.4,5.40,2.0,4.2,4.8,Wallet,Heavy,Clear,5,7,37,19
1,T000002,R11395,D03557,2025-01-16 06:05:51,2025-01-16 06:08:51.000,0.6,180.0,3.20,2.0,4.1,4.6,Cash,Heavy,Rain,3,6,87,58
2,T000003,R48556,D17160,2025-01-13 04:53:06,2025-01-13 05:03:16.000,4.8,610.0,9.20,1.0,4.5,4.6,Wallet,Moderate,Clear,0,4,24,34
3,T000004,R37066,D00996,2025-03-07 00:24:14,2025-03-07 00:29:56.000,3.1,342.0,7.58,1.6,4.4,4.9,Card,Light,Clear,4,0,3,222
4,T000005,R39100,D08572,2024-10-14 02:46:37,2024-10-14 03:03:19.000,7.0,1002.0,15.30,1.8,4.4,4.5,Cash,Moderate,Clear,0,2,70,107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9999995,T9999996,R25062,D13498,2025-03-27 12:10:50,2025-03-27 12:33:29.600,7.8,1359.6,17.21,1.9,4.3,4.8,Card,Light,Rain,3,12,153,156
9999996,T9999997,R23411,D00948,2024-11-02 20:35:57,2024-11-02 20:46:01.000,5.2,604.0,11.10,1.5,4.7,4.3,Card,Moderate,Clear,5,20,123,7
9999997,T9999998,R04063,D01255,2024-11-21 08:41:02,2024-11-21 08:58:26.000,5.2,1044.0,11.62,1.7,4.2,4.8,Card,Moderate,Rain,3,8,54,90
9999998,T9999999,R16639,D09822,2024-12-27 17:12:46,2024-12-27 17:26:54.000,3.2,848.0,6.80,1.0,4.6,4.8,Card,Light,Clear,4,17,101,155


In [ ]:
df.to_csv("swiftride_10M.csv")

In [15]:
df_csv = pd.read_csv("swiftride_10M.csv")
df_csv

,Unnamed: 0,trip_id,rider_id,driver_id,trip_start_time,trip_end_time,trip_distance,trip_duration,fare_amount,surge_multiplier,driver_rating,rider_rating,payment_method,traffic_condition,weather,day_of_week,hour_of_day,driver_tenure_days,rider_tenure_days
0,0,T000001,R00770,D02434,2025-03-22 07:40:22,2025-03-22 07:44:21.400,1.7,239.4,5.40,2.0,4.2,4.8,Wallet,Heavy,Clear,5,7,37,19
1,1,T000002,R11395,D03557,2025-01-16 06:05:51,2025-01-16 06:08:51.000,0.6,180.0,3.20,2.0,4.1,4.6,Cash,Heavy,Rain,3,6,87,58
2,2,T000003,R48556,D17160,2025-01-13 04:53:06,2025-01-13 05:03:16.000,4.8,610.0,9.20,1.0,4.5,4.6,Wallet,Moderate,Clear,0,4,24,34
3,3,T000004,R37066,D00996,2025-03-07 00:24:14,2025-03-07 00:29:56.000,3.1,342.0,7.58,1.6,4.4,4.9,Card,Light,Clear,4,0,3,222
4,4,T000005,R39100,D08572,2024-10-14 02:46:37,2024-10-14 03:03:19.000,7.0,1002.0,15.30,1.8,4.4,4.5,Cash,Moderate,Clear,0,2,70,107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9999995,9999995,T9999996,R25062,D13498,2025-03-27 12:10:50,2025-03-27 12:33:29.600,7.8,1359.6,17.21,1.9,4.3,4.8,Card,Light,Rain,3,12,153,156
9999996,9999996,T9999997,R23411,D00948,2024-11-02 20:35:57,2024-11-02 20:46:01.000,5.2,604.0,11.10,1.5,4.7,4.3,Card,Moderate,Clear,5,20,123,7
9999997,9999997,T9999998,R04063,D01255,2024-11-21 08:41:02,2024-11-21 08:58:26.000,5.2,1044.0,11.62,1.7,4.2,4.8,Card,Moderate,Rain,3,8,54,90
9999998,9999998,T9999999,R16639,D09822,2024-12-27 17:12:46,2024-12-27 17:26:54.000,3.2,848.0,6.80,1.0,4.6,4.8,Card,Light,Clear,4,17,101,155
